# Урок 10 — Ітератори й генератори

> **Сценарій:** у нас з'явився `DataFrame` із **500 000 рядків** синтетичних замовлень ресторану. Написати аналітику по днях тижня й типу прийому їжі "в лоб" — це 200 рядків нечитабельного коду, який ще й повільно і незручно тестувати. Розв'яжемо це грамотною архітектурою: чисті функції + **пайплайн з генераторів**, який не тримає всі 500 000 записів у пам'яті одночасно.

**Модулі цього уроку:** `datetime` (`.hour`, `.strftime`), `typing.NamedTuple`, `collections` (`Counter`, `defaultdict`), `sys.getsizeof` (вимірювання пам'яті).

## 🔁 RETRIEVE — пригадай попередні уроки (без підглядання)

1. Що робить `d.get(key, 0) + 1`, і для чого цей прийом потрібен?
2. Чим `NamedTuple` відрізняється від звичайного `tuple`?
3. Чому `[["."] * SIZE] * SIZE` — небезпечний запис для списку списків (Урок 6)?

Сьогоднішня тема виростає з практичної проблеми: коли даних стає багато (500 000+ записів), звичайний список перестає вміщатись у пам'ять — знадобляться **генератори**.

## 📖 CONCEPT

### 1. Ітератор: що насправді робить `for`

Коли ти пишеш `for x in [1, 2, 3]:`, Python під капотом робить дві речі: викликає `iter()` на список, отримуючи **ітератор**, а потім повторно викликає `next()` на цьому ітераторі, поки той не підніме `StopIteration`. `for` — це просто зручна обгортка над цим протоколом.

In [1]:
numbers = [10, 20, 30]

# iter() перетворює ітерований об'єкт (list) на ітератор
it = iter(numbers)
print(f"Тип ітератора: {type(it)}")

# next() дістає по одному значенню, "просуваючи" ітератор
print(next(it))   # 10
print(next(it))   # 20
print(next(it))   # 30

# Значення скінчились -- next() підіймає StopIteration
try:
    next(it)
except StopIteration:
    print("StopIteration: значення скінчились")

print()
print("Саме це робить for numbers: iter() один раз, next() у циклі, доки не StopIteration")

Тип ітератора: <class 'list_iterator'>
10
20
30
StopIteration: значення скінчились

Саме це робить for numbers: iter() один раз, next() у циклі, доки не StopIteration


### 2. `yield` — функція, що ставиться на паузу

Звичайна функція з `return` віддає значення один раз і завершується. Функція з `yield` **призупиняється** на кожному `yield`, "заморожуючи" весь свій стан (локальні змінні, позицію в коді), і продовжує з того самого місця при наступному `next()`. Виклик такої функції не виконує її код одразу — він повертає **об'єкт-генератор**, "рецепт", а не результат:

In [2]:
def count_up_to(n):
    """Генератор: віддає числа від 1 до n, по одному."""
    i = 1
    while i <= n:
        yield i     # призупиняється тут, "запам'ятовує" i
        i += 1


gen = count_up_to(3)
print(f"Тип: {type(gen)}")
print("Виклик count_up_to(3) НЕ виконав жодного рядка тіла функції -- лише створив генератор.")
print()

print(next(gen))   # тепер виконується до першого yield -> 1
print(next(gen))   # продовжує з місця зупинки -> 2
print(next(gen))   # -> 3

try:
    next(gen)
except StopIteration:
    print("StopIteration: while i <= n стало хибним, функція завершилась")

print()
print("for теж працює з генератором напряму -- бо генератор сам є ітератором:")
for value in count_up_to(3):
    print(" ", value)

Тип: <class 'generator'>
Виклик count_up_to(3) НЕ виконав жодного рядка тіла функції -- лише створив генератор.

1
2
3
StopIteration: while i <= n стало хибним, функція завершилась

for теж працює з генератором напряму -- бо генератор сам є ітератором:
  1
  2
  3


### 3. Навіщо це потрібно: генератор не зберігає всі значення одразу

`count_up_to(3)` — тривіальний приклад, різниця непомітна. Але уяви `count_up_to(500_000)`: список `[1, 2, ..., 500000]` займає реальну пам'ять під усі 500 000 чисел одразу. Генератор `count_up_to(500_000)` у кожен момент часу тримає в пам'яті лише **поточне значення** `i` — байдуже, рахуємо ми до 3 чи до 500 000.

Це саме та проблема, яку зараз розв'яжемо на реальних даних:

```text
DataFrame (500 000 рядків)
      ↓
Generator[RawOrder]   ← адаптер (генератор, не list)
      ↓                 datetime, typing.NamedTuple
Generator[Order]      ← трансформер (ще генератор)
      ↓                 datetime.strftime, .hour
list[Order]           ← матеріалізуємо лише коли треба аналітика
      ↓
analytics             ← collections.Counter, defaultdict
```

## 🛠️ CREATE

### Крок 1 — дані: 500 000 синтетичних замовлень

Генеруємо `DataFrame` на основі реального датасету `seaborn.tips` (стать, курець/ні, розмір столика — беремо звідти), з випадковим `timestamp` за два роки:

In [3]:
import seaborn as sns
import pandas as pd
import random
from datetime import datetime, timedelta

random.seed(7)

tips = sns.load_dataset("tips")
sex_values    = tips["sex"].values
smoker_values = tips["smoker"].values
size_values   = tips["size"].values

rows = []
start = datetime(2023, 1, 1)
end   = datetime(2025, 1, 1)
delta = end - start

N = 500_000

for i in range(N):
    random_seconds = random.randint(0, int(delta.total_seconds()))
    timestamp = start + timedelta(seconds=random_seconds)
    bill = round(random.uniform(10, 60), 2)
    tip  = round(bill * random.uniform(0.10, 0.25), 2)
    rows.append({
        "total_bill": bill,
        "tip":        tip,
        "sex":        random.choice(sex_values),
        "smoker":     random.choice(smoker_values),
        "size":       random.choice(size_values),
        "timestamp":  timestamp,
    })

df = pd.DataFrame(rows)
print(df.head())
print(f"\nВсього рядків: {len(df):,}")
print("Є timestamp, але немає окремих колонок 'day' і 'time' -- їх треба витягти.")

   total_bill   tip   sex smoker  size           timestamp
0       57.39  9.14  Male     No     3 2023-09-09 12:40:48
1       14.71  2.76  Male     No     4 2024-02-21 05:33:52
2       14.30  2.33  Male     No     6 2023-01-30 02:58:11
3       12.96  2.40  Male     No     2 2023-11-26 17:36:40
4       39.15  4.28  Male    Yes     2 2024-05-02 08:12:10

Всього рядків: 500,000
Є timestamp, але немає окремих колонок 'day' і 'time' -- їх треба витягти.


### Крок 2 — проблема: код без архітектури

Уявімо, що ми одразу пишемо аналітику "в лоб", все в одному циклі (демонструємо тільки на 10 000 рядках -- на всіх 500 000 цей стиль був би ще повільнішим і нечитабельнішим):

In [4]:
# ❌ БЕЗ АРХІТЕКТУРИ -- все в одному монструозному циклі

rev_by_day  = {}
cnt_by_day  = {}
tips_by_day = {}

for _, row in df.head(10_000).iterrows():
    ts = row["timestamp"]
    # Логіка визначення дня -- прямо в циклі
    day = ts.strftime("%a")
    # Логіка визначення прийому їжі -- теж тут
    hour = ts.hour
    if 11 <= hour <= 15:
        meal = "Lunch"
    elif 17 <= hour <= 23:
        meal = "Dinner"
    else:
        meal = "Other"
    # Підрахунок -- теж тут
    if day not in rev_by_day:
        rev_by_day[day]  = 0.0
        cnt_by_day[day]  = 0
        tips_by_day[day] = 0.0
    rev_by_day[day]  += float(row["total_bill"])
    cnt_by_day[day]  += 1
    tips_by_day[day] += float(row["tip"])

print("Результат є, але код:")
print("  -- не можна протестувати окремо логіку дня/прийому їжі")
print("  -- не можна перевикористати в іншому місці")
print("  -- для 500 000 рядків -- повільно і важко читати")
print(f"  -- всього {sum(cnt_by_day.values())} рядків оброблено")

Результат є, але код:
  -- не можна протестувати окремо логіку дня/прийому їжі
  -- не можна перевикористати в іншому місці
  -- для 500 000 рядків -- повільно і важко читати
  -- всього 10000 рядків оброблено


**Сигнал тривоги (code smell):** бізнес-логіка (`day`, `meal`) перемішана з читанням даних, а підрахунок перемішаний з трансформацією. Розкладаємо на шари:

1. **Domain model** (`NamedTuple`) — визначаємо структури даних.
2. **Чисті функції** (`datetime`) — логіка трансформації окремо, її можна тестувати.
3. **Generator-адаптер** — `DataFrame` → `Generator[RawOrder]`.
4. **Generator-трансформер** — `RawOrder` → `Order`.
5. **Аналітика** (`Counter`, `defaultdict`) — окремим шаром, над готовими даними.

### Крок 3 — Domain model: `RawOrder` і `Order`

Два окремі `NamedTuple`: `RawOrder` відповідає тому, що прийшло з `DataFrame` (є `timestamp`, немає `day`/`time`); `Order` -- уже збагачена версія (`day` і `time` витягнуті, `timestamp` більше не потрібен):

In [5]:
from typing import NamedTuple


class RawOrder(NamedTuple):
    total_bill: float
    tip:        float
    sex:        str
    smoker:     str
    size:       int
    timestamp:  datetime


class Order(NamedTuple):
    total_bill: float
    tip:        float
    sex:        str
    smoker:     str
    day:        str
    time:       str
    size:       int


sample_raw = RawOrder(
    total_bill=35.50, tip=7.10, sex="Male", smoker="No", size=2,
    timestamp=datetime(2024, 3, 15, 19, 30),   # п'ятниця, 19:30
)

print(f"RawOrder:            {sample_raw}")
print(f"Доступ по імені:     sample_raw.sex = {sample_raw.sex}")
print(f"Доступ по індексу:   sample_raw[2]  = {sample_raw[2]}")
print(f"timestamp.hour:      {sample_raw.timestamp.hour}")

try:
    sample_raw.sex = "Female"
except AttributeError as e:
    print(f"AttributeError: {e}  <- NamedTuple незмінний, як і звичайний tuple")

RawOrder:            RawOrder(total_bill=35.5, tip=7.1, sex='Male', smoker='No', size=2, timestamp=datetime.datetime(2024, 3, 15, 19, 30))
Доступ по імені:     sample_raw.sex = Male
Доступ по індексу:   sample_raw[2]  = Male
timestamp.hour:      19
AttributeError: can't set attribute  <- NamedTuple незмінний, як і звичайний tuple


### Крок 4 — чисті функції з `datetime`

Функції без побічних ефектів: приймають параметр, повертають результат, нічого більше не змінюють — це і робить їх легкими для незалежного тестування:

In [6]:
def meal_type_from_hour(hour: int) -> str:
    """Тип прийому їжі за годиною доби. Чиста функція."""
    if 11 <= hour <= 15:
        return "Lunch"
    if 17 <= hour <= 23:
        return "Dinner"
    return "Other"


def day_from_timestamp(ts: datetime) -> str:
    """Скорочена назва дня тижня. strftime('%a') -> Mon, Tue, ..."""
    return ts.strftime("%a")


test_times = [(8, "ранок"), (12, "полудень"), (14, "обід"), (16, "мертва зона"), (19, "вечеря")]
print("meal_type_from_hour:")
for hour, label in test_times:
    print(f"  hour={hour:2d} ({label:<12}) -> {meal_type_from_hour(hour)}")

print()
print("day_from_timestamp:")
for d in [datetime(2024, 3, 11), datetime(2024, 3, 15), datetime(2024, 3, 17)]:
    print(f"  {d.strftime('%Y-%m-%d')} ({d.strftime('%A'):<10}) -> {day_from_timestamp(d)}")

assert meal_type_from_hour(12) == "Lunch"
assert meal_type_from_hour(20) == "Dinner"
assert meal_type_from_hour(2)  == "Other"
print("\nOK -- усі три гілки meal_type_from_hour перевірено")

meal_type_from_hour:
  hour= 8 (ранок       ) -> Other
  hour=12 (полудень    ) -> Lunch
  hour=14 (обід        ) -> Lunch
  hour=16 (мертва зона ) -> Other
  hour=19 (вечеря      ) -> Dinner

day_from_timestamp:
  2024-03-11 (Monday    ) -> Mon
  2024-03-15 (Friday    ) -> Fri
  2024-03-17 (Sunday    ) -> Sun

OK -- усі три гілки meal_type_from_hour перевірено


### Крок 5 — адаптер: `DataFrame` → генератор

**Чому генератор, а не list comprehension?** List comprehension будує весь список одразу -- для 500 000 об'єктів це реальна пам'ять. Генератор віддає по одному об'єкту й "забуває" про нього одразу після використання. Спочатку -- eager-версія (list), для порівняння, потім -- generator-версія (`yield`):

In [7]:
def raw_orders_as_list(df):
    """Eager-версія адаптера -- будує весь список одразу (для порівняння)."""
    return [
        RawOrder(
            total_bill=float(row["total_bill"]), tip=float(row["tip"]),
            sex=str(row["sex"]), smoker=str(row["smoker"]),
            size=int(row["size"]), timestamp=row["timestamp"],
        )
        for _, row in df.iterrows()
    ]


def raw_orders_from_df(df):
    """Generator-адаптер: DataFrame -> RawOrder по одному.
    yield замість return -- весь список НЕ зберігається в пам'яті.
    """
    for _, row in df.iterrows():
        yield RawOrder(
            total_bill=float(row["total_bill"]), tip=float(row["tip"]),
            sex=str(row["sex"]), smoker=str(row["smoker"]),
            size=int(row["size"]), timestamp=row["timestamp"],
        )


gen = raw_orders_from_df(df.head(5))
print(f"Тип: {type(gen)}")
print("Перші 3 елементи (кожен next() дає один):")
print(f"  {next(gen)}")
print(f"  {next(gen)}")
print(f"  {next(gen)}")
print()
print("Генератор -- це 'рецепт', а не 'список'. Дані генеруються ліниво, по одному.")

Тип: <class 'generator'>
Перші 3 елементи (кожен next() дає один):
  RawOrder(total_bill=57.39, tip=9.14, sex='Male', smoker='No', size=3, timestamp=Timestamp('2023-09-09 12:40:48'))
  RawOrder(total_bill=14.71, tip=2.76, sex='Male', smoker='No', size=4, timestamp=Timestamp('2024-02-21 05:33:52'))
  RawOrder(total_bill=14.3, tip=2.33, sex='Male', smoker='No', size=6, timestamp=Timestamp('2023-01-30 02:58:11'))

Генератор -- це 'рецепт', а не 'список'. Дані генеруються ліниво, по одному.


### Крок 6 — генератор проти list comprehension: реальний вимір пам'яті

Не повіримо на слово -- виміряємо `sys.getsizeof()` обох варіантів:

In [8]:
import sys

list_comp = [
    RawOrder(float(row["total_bill"]), float(row["tip"]), str(row["sex"]),
             str(row["smoker"]), int(row["size"]), row["timestamp"])
    for _, row in df.head(10_000).iterrows()
]

gen_expr = (
    RawOrder(float(row["total_bill"]), float(row["tip"]), str(row["sex"]),
              str(row["smoker"]), int(row["size"]), row["timestamp"])
    for _, row in df.head(10_000).iterrows()
)

list_size = sys.getsizeof(list_comp)
gen_size  = sys.getsizeof(gen_expr)

print(f"list[10k]:  {list_size:>12,} байт  <- весь список у RAM")
print(f"generator:  {gen_size:>12,} байт  <- лише стан машини")
print(f"Різниця:    ~x{list_size // gen_size}")

assert gen_size < list_size, "Генератор мав би займати менше пам'яті, ніж матеріалізований список"

# Перевіримо, що розмір генератора НЕ залежить від кількості елементів, які він видасть:
gen_expr_small = (RawOrder(0.0, 0.0, "x", "x", 1, df["timestamp"].iloc[0]) for _ in range(10))
gen_expr_big   = (RawOrder(0.0, 0.0, "x", "x", 1, df["timestamp"].iloc[0]) for _ in range(500_000))
size_small = sys.getsizeof(gen_expr_small)
size_big   = sys.getsizeof(gen_expr_big)
print(f"\ngenerator (розрахований на 10 елементів):      {size_small} байт")
print(f"generator (розрахований на 500 000 елементів): {size_big} байт")
assert size_small == size_big, "Розмір генератора не повинен залежати від того, скільки він видасть"
print("OK -- розмір генератора не залежить від N, розмір списку -- залежить лінійно")

list[10k]:        85,176 байт  <- весь список у RAM
generator:           272 байт  <- лише стан машини
Різниця:    ~x313

generator (розрахований на 10 елементів):      256 байт
generator (розрахований на 500 000 елементів): 256 байт
OK -- розмір генератора не залежить від N, розмір списку -- залежить лінійно


### Крок 7 — трансформер: `RawOrder` → `Order`

Ще одна чиста функція: збагачує `RawOrder`, витягуючи `day` і `time` з `timestamp` через уже готові `day_from_timestamp` і `meal_type_from_hour`:

In [9]:
def enrich_order(raw: RawOrder) -> Order:
    """Трансформер: RawOrder -> Order. Витягує day і time з timestamp."""
    return Order(
        total_bill=raw.total_bill, tip=raw.tip, sex=raw.sex, smoker=raw.smoker,
        day=day_from_timestamp(raw.timestamp),
        time=meal_type_from_hour(raw.timestamp.hour),
        size=raw.size,
    )


test_raw = RawOrder(42.80, 8.56, "Female", "Yes", 3, datetime(2024, 6, 22, 13, 15))
enriched = enrich_order(test_raw)

print(f"ВХІД:  timestamp = {test_raw.timestamp}  (субота, 13:15)")
print(f"ВИХІД: day = {enriched.day}, time = {enriched.time}")
print(f"Повний Order: {enriched}")

assert enriched.day == "Sat"
assert enriched.time == "Lunch"
print("\nOK")

ВХІД:  timestamp = 2024-06-22 13:15:00  (субота, 13:15)
ВИХІД: day = Sat, time = Lunch
Повний Order: Order(total_bill=42.8, tip=8.56, sex='Female', smoker='Yes', day='Sat', time='Lunch', size=3)

OK


### Крок 8 — з'єднуємо генератори в пайплайн

`enrich_orders` — теж генератор: тягне з вхідного генератора по одному `RawOrder`, віддає по одному `Order`. У будь-який момент у пам'яті — лише один об'єкт на кожному з двох рівнів:

In [10]:
def enrich_orders(raw_orders):
    """Generator-трансформер: тягне RawOrder по одному, віддає Order."""
    for raw in raw_orders:
        yield enrich_order(raw)


df_sample = df.head(5)
raw_gen    = raw_orders_from_df(df_sample)    # лінивий
orders_gen = enrich_orders(raw_gen)           # теж лінивий

print(f"raw_gen:    {type(raw_gen)}")
print(f"orders_gen: {type(orders_gen)}")
print("До list() -- нічого ще не обраховано.")
print()

sample_orders = list(orders_gen)   # ось тут генератори фактично запускаються
print(f"list(orders_gen) -> {len(sample_orders)} Order-об'єктів")
for o in sample_orders:
    print(f"  {o.day:5} | {o.time:<7} | ${o.total_bill:.2f}")

raw_gen:    <class 'generator'>
orders_gen: <class 'generator'>
До list() -- нічого ще не обраховано.

list(orders_gen) -> 5 Order-об'єктів
  Sat   | Lunch   | $57.39
  Wed   | Other   | $14.71
  Mon   | Other   | $14.30
  Sun   | Dinner  | $12.96
  Thu   | Other   | $39.15


### Крок 9 — матеріалізація: коли й навіщо

Генератор можна пройти **лише один раз**. Аналітика зазвичай потребує кількох проходів (`sum()`, потім `Counter()`, потім `sorted()`) — тому матеріалізуємо **один раз** у `list`, а далі працюємо з ним скільки завгодно:

In [11]:
raw_gen    = raw_orders_from_df(df)
orders_gen = enrich_orders(raw_gen)

orders = list(orders_gen)   # єдина матеріалізація на весь пайплайн

print(f"orders: {len(orders):,} Order-об'єктів")
print(f"Перший:   {orders[0]}")
print(f"Останній: {orders[-1]}")

assert len(orders) == len(df)
print("\nOK -- усі 500 000 рядків пройшли через пайплайн")

orders: 500,000 Order-об'єктів
Перший:   Order(total_bill=57.39, tip=9.14, sex='Male', smoker='No', day='Sat', time='Lunch', size=3)
Останній: Order(total_bill=10.16, tip=2.43, sex='Female', smoker='Yes', day='Fri', time='Other', size=2)

OK -- усі 500 000 рядків пройшли через пайплайн


### Крок 10 — доказ, а не обіцянка: eager і generator-пайплайн дають ІДЕНТИЧНИЙ результат

Пайплайн з генераторів обробляє дані інакше (ліниво, по одному), ніж eager-версія зі списками. Це не повинно міняти РЕЗУЛЬТАТ — лише спосіб його отримання. Перевіримо це прямим `assert`, а не голослівно, на підмножині даних (щоб eager-версія не з'їла зайву пам'ять під час самої перевірки):

In [12]:
df_check = df.head(20_000)

# Eager-шлях: list comprehension на кожному кроці
raw_list    = raw_orders_as_list(df_check)
orders_eager = [enrich_order(r) for r in raw_list]

# Generator-шлях: той самий пайплайн, але ліниво
orders_lazy = list(enrich_orders(raw_orders_from_df(df_check)))

assert orders_eager == orders_lazy, "Generator-пайплайн дав інший результат, ніж eager-версія!"
print(f"OK -- {len(orders_eager):,} Order-об'єктів, eager і generator шляхи побайтово однакові")

# І аналітика поверх обох -- теж однакова
from statistics import mean
avg_eager = mean(o.total_bill for o in orders_eager)
avg_lazy  = mean(o.total_bill for o in orders_lazy)
assert avg_eager == avg_lazy
print(f"OK -- середній чек однаковий в обох варіантах: ${avg_eager:.2f}")

OK -- 20,000 Order-об'єктів, eager і generator шляхи побайтово однакові
OK -- середній чек однаковий в обох варіантах: $35.00


### Крок 11 — аналітика: `Counter` і `defaultdict`

Тепер, коли `orders` матеріалізовано (усі 500 000), рахуємо аналітику поверх звичайного списку -- аналітичні функції нічого не знають про те, що дані колись пройшли через генератори:

In [13]:
from collections import Counter, defaultdict


def avg_bill(orders_: list) -> float:
    return sum(o.total_bill for o in orders_) / len(orders_)


def busiest_day(orders_: list) -> str:
    counts = Counter(o.day for o in orders_)
    return counts.most_common(1)[0][0]


def group_by_day(orders_: list) -> dict:
    result = defaultdict(lambda: {"count": 0, "revenue": 0.0, "tips": 0.0})
    for o in orders_:
        result[o.day]["count"]   += 1
        result[o.day]["revenue"] += o.total_bill
        result[o.day]["tips"]    += o.tip
    return {
        day: {
            "count": v["count"],
            "revenue": round(v["revenue"], 2),
            "avg_bill": round(v["revenue"] / v["count"], 2),
            "avg_tip_pct": round((v["tips"] / v["revenue"]) * 100, 1),
        }
        for day, v in sorted(result.items(), key=lambda x: -x[1]["revenue"])
    }


def top_tips(orders_: list, n: int = 5) -> list:
    return sorted(orders_, key=lambda o: o.tip / o.total_bill, reverse=True)[:n]


day_counter = Counter(o.day for o in orders)
print("Counter по днях тижня:")
for day, count in day_counter.most_common(3):
    print(f"  {day}: {count:,} замовлень")

print(f"\nСередній чек: ${avg_bill(orders):.2f}")
print(f"Найзавантаженіший день: {busiest_day(orders)}")

Counter по днях тижня:
  Sun: 71,804 замовлень
  Mon: 71,600 замовлень
  Tue: 71,506 замовлень

Середній чек: $34.98
Найзавантаженіший день: Sun


### Крок 12 — повний звіт

In [14]:
print("=" * 55)
print(f"  АНАЛІТИКА РЕСТОРАНУ: {len(orders):,} замовлень")
print("=" * 55)
print(f"\n  Середній чек: ${avg_bill(orders):.2f}")
print(f"  Найзавантаженіший день: {busiest_day(orders)}")

print(f"\n  По днях тижня (sorted by revenue):")
print(f"  {'День':<6} {'Чеків':>10} {'Виручка':>14} {'Avg чек':>10} {'Tip%':>6}")
for day, stats in group_by_day(orders).items():
    print(f"  {day:<6} {stats['count']:>10,} ${stats['revenue']:>12,.2f} "
          f"${stats['avg_bill']:>8.2f} {stats['avg_tip_pct']:>5.1f}%")

print(f"\n  Топ-5 за % чайових:")
for i, o in enumerate(top_tips(orders, n=5), 1):
    pct = round(o.tip / o.total_bill * 100, 1)
    print(f"  {i}. {o.day:<6} {o.time:<8} ${o.total_bill:>8.2f} ${o.tip:>7.2f} {pct:>5.1f}%")

  АНАЛІТИКА РЕСТОРАНУ: 500,000 замовлень

  Середній чек: $34.98


  Найзавантаженіший день: Sun

  По днях тижня (sorted by revenue):
  День        Чеків        Виручка    Avg чек   Tip%


  Sun        71,804 $2,514,913.85 $   35.02  17.5%
  Mon        71,600 $2,506,315.92 $   35.00  17.5%
  Tue        71,506 $2,503,643.35 $   35.01  17.5%
  Fri        71,384 $2,496,726.61 $   34.98  17.5%
  Wed        71,288 $2,495,754.41 $   35.01  17.5%
  Thu        71,459 $2,492,564.68 $   34.88  17.5%
  Sat        70,959 $2,480,641.24 $   34.96  17.5%

  Топ-5 за % чайових:


  1. Fri    Other    $   10.07 $   2.52  25.0%
  2. Mon    Dinner   $   10.39 $   2.60  25.0%
  3. Fri    Other    $   10.63 $   2.66  25.0%
  4. Thu    Dinner   $   10.71 $   2.68  25.0%
  5. Sun    Lunch    $   10.83 $   2.71  25.0%


### Крок 13 — архітектура дозволяє легко розширювати

Кожне нове зрізання даних -- окрема чиста функція, решта пайплайну не змінюється:

In [15]:
def group_by_meal(orders_: list) -> dict:
    result = defaultdict(lambda: {"count": 0, "revenue": 0.0, "tips": 0.0})
    for o in orders_:
        result[o.time]["count"]   += 1
        result[o.time]["revenue"] += o.total_bill
        result[o.time]["tips"]    += o.tip
    return {
        meal: {
            "count": v["count"],
            "avg_bill": round(v["revenue"] / v["count"], 2),
            "avg_tip_pct": round((v["tips"] / v["revenue"]) * 100, 1),
        }
        for meal, v in sorted(result.items(), key=lambda x: -x[1]["count"])
    }


print("По типу прийому їжі:")
for meal, stats in group_by_meal(orders).items():
    print(f"  {meal:<8} {stats['count']:>8,} чеків | avg ${stats['avg_bill']:.2f} | tip {stats['avg_tip_pct']:.1f}%")

print("\nДодавання нової аналітики = одна нова функція. Решта коду не змінюється.")

По типу прийому їжі:


  Other     250,223 чеків | avg $34.95 | tip 17.5%
  Dinner    145,538 чеків | avg $35.03 | tip 17.5%
  Lunch     104,239 чеків | avg $34.99 | tip 17.5%

Додавання нової аналітики = одна нова функція. Решта коду не змінюється.


### Шпаргалка

```text
iter(x) / next(it)      -- ручний протокол ітерації, той самий, що всередині for
StopIteration            -- ітератор вичерпано

def gen():
    yield value           -- функція стає генератором; призупиняється на yield

generator vs list:
  list      -- всі елементи одразу в RAM, можна пройти кілька разів
  generator -- по одному елементу, ~const пам'ять, лише ОДИН прохід

Матеріалізація (list(gen)) -- один раз, коли потрібно кілька проходів (аналітика)

Pipeline:
  DataFrame -> Generator[RawOrder] -> Generator[Order] -> list[Order] -> Counter/defaultdict
```

## 🔄 TRANSFER

Самостійні завдання на тому самому пайплайні. Кожне -- окрема клітинка з готовим еталонним розв'язком; спробуй спочатку сам, потім звір.

### Завдання 1 -- `group_by_month`

Додай до збагаченого запису поле `month` (`"YYYY-MM"`, через `timestamp.strftime("%Y-%m")`) і напиши агрегацію по місяцях.

In [16]:
# YOUR CODE HERE
# BEGIN SOLUTION
class OrderMonth(NamedTuple):
    total_bill: float
    tip:        float
    day:        str
    month:      str


def enrich_order_month(raw: RawOrder) -> OrderMonth:
    return OrderMonth(
        total_bill=raw.total_bill, tip=raw.tip,
        day=day_from_timestamp(raw.timestamp),
        month=raw.timestamp.strftime("%Y-%m"),
    )


def group_by_month(orders_: list) -> dict:
    result = defaultdict(lambda: {"count": 0, "revenue": 0.0})
    for o in orders_:
        result[o.month]["count"]   += 1
        result[o.month]["revenue"] += o.total_bill
    return {m: {"count": v["count"], "revenue": round(v["revenue"], 2)}
            for m, v in sorted(result.items())}
# END SOLUTION

month_orders = [enrich_order_month(r) for r in raw_orders_from_df(df.head(50_000))]
monthly = group_by_month(month_orders)
print(f"Місяців у вибірці: {len(monthly)}")
first_month = sorted(monthly)[0]
print(f"{first_month}: {monthly[first_month]}")

assert len(monthly) > 1
assert all("count" in v and "revenue" in v for v in monthly.values())
print("OK")

Місяців у вибірці: 24
2023-01: {'count': 2134, 'revenue': 74901.1}
OK


### Завдання 2 -- `filter_orders` (generator)

Генератор-фільтр: пропускає лише елементи вхідного генератора, які задовольняють `predicate`, без матеріалізації.

In [17]:
# YOUR CODE HERE
# BEGIN SOLUTION
def filter_orders(orders_gen, predicate):
    """Generator-фільтр: пропускає лише Order, де predicate(o) == True."""
    for o in orders_gen:
        if predicate(o):
            yield o
# END SOLUTION

raw_gen_t2    = raw_orders_from_df(df.head(50_000))
orders_gen_t2 = enrich_orders(raw_gen_t2)
dinner_gen    = filter_orders(orders_gen_t2, lambda o: o.time == "Dinner")

assert hasattr(dinner_gen, "__next__"), "filter_orders має лишатись генератором (yield, не list)"
dinners = list(dinner_gen)
print(f"Вечеря: {len(dinners):,} замовлень із 50 000")
assert all(o.time == "Dinner" for o in dinners)
print("OK")

Вечеря: 14,393 замовлень із 50 000
OK


### Завдання 3 -- `top_days_by_tips`

Топ `n` днів тижня за **сумарними** чайовими (не середніми).

In [18]:
# YOUR CODE HERE
# BEGIN SOLUTION
def top_days_by_tips(orders_: list, n: int = 3) -> list:
    """Топ n днів за сумарними чайовими: [(day, total_tips), ...] за спаданням."""
    tips_by_day_ = defaultdict(float)
    for o in orders_:
        tips_by_day_[o.day] += o.tip
    return sorted(tips_by_day_.items(), key=lambda x: x[1], reverse=True)[:n]
# END SOLUTION

top3 = top_days_by_tips(orders, n=3)
for day, tips_sum in top3:
    print(f"{day}: ${tips_sum:,.2f}")

assert len(top3) == 3
assert top3[0][1] >= top3[1][1] >= top3[2][1]
print("OK")

Sun: $439,523.48
Mon: $438,195.45
Tue: $437,747.53
OK


### Завдання 4 -- `RichOrder`

Розширений запис із `tip_pct` (відсоток чайових) і `bill_per_person` (рахунок на людину), плюс генераторний трансформер.

In [19]:
# YOUR CODE HERE
# BEGIN SOLUTION
class RichOrder(NamedTuple):
    total_bill:      float
    tip:             float
    sex:             str
    smoker:          str
    day:             str
    time:            str
    size:            int
    tip_pct:         float
    bill_per_person: float


def enrich_to_rich(order: Order) -> RichOrder:
    return RichOrder(
        total_bill=order.total_bill, tip=order.tip, sex=order.sex, smoker=order.smoker,
        day=order.day, time=order.time, size=order.size,
        tip_pct=round(order.tip / order.total_bill * 100, 1),
        bill_per_person=round(order.total_bill / order.size, 2),
    )


def enrich_to_rich_gen(orders_gen):
    for o in orders_gen:
        yield enrich_to_rich(o)
# END SOLUTION

rich_sample = list(enrich_to_rich_gen(enrich_orders(raw_orders_from_df(df.head(5)))))
for r in rich_sample:
    print(f"${r.total_bill:.2f} | tip {r.tip_pct}% | на людину ${r.bill_per_person:.2f}")

assert all(0 <= r.tip_pct <= 100 for r in rich_sample)
assert all(r.bill_per_person == round(r.total_bill / r.size, 2) for r in rich_sample)
print("OK")

$57.39 | tip 15.9% | на людину $19.13
$14.71 | tip 18.8% | на людину $3.68
$14.30 | tip 16.3% | на людину $2.38
$12.96 | tip 18.5% | на людину $6.48
$39.15 | tip 10.9% | на людину $19.57
OK


### Завдання 5* -- `run_pipeline`

Складає весь пайплайн у одну функцію з опційним обмеженням рядків і фільтром по `time`.

In [20]:
# YOUR CODE HERE
# BEGIN SOLUTION
def run_pipeline(df_, max_rows=None, time_filter=None) -> dict:
    working_df = df_.head(max_rows) if max_rows is not None else df_
    raw_gen_ = raw_orders_from_df(working_df)
    orders_gen_ = enrich_orders(raw_gen_)
    if time_filter is not None:
        orders_gen_ = filter_orders(orders_gen_, lambda o: o.time == time_filter)
    orders_local = list(orders_gen_)
    return {
        "count": len(orders_local),
        "avg_bill": round(avg_bill(orders_local), 2) if orders_local else 0.0,
        "by_day": group_by_day(orders_local),
    }
# END SOLUTION

result = run_pipeline(df, max_rows=100_000, time_filter="Dinner")
print(f"count: {result['count']:,}")
print(f"avg_bill: ${result['avg_bill']}")

assert result["count"] <= 100_000
assert isinstance(result["by_day"], dict)
print("OK")

count: 28,865
avg_bill: $35.02
OK


## ✅ Самоперевірка (5 запитань)

**1.** Що конкретно робить `for` "під капотом" зі списком?

<details><summary>Відповідь</summary>Викликає <code>iter()</code> на список один раз, отримуючи ітератор, а потім повторно <code>next()</code> на ньому, доки не отримає <code>StopIteration</code>.</details>

**2.** Чому `sys.getsizeof()` генератора не росте разом із кількістю елементів, які він видасть?

<details><summary>Відповідь</summary>Генератор зберігає лише свій поточний стан виконання (де він "зупинився"), а не самі майбутні значення -- вони обчислюються по одному, коли їх запитують через <code>next()</code>.</details>

**3.** Генератор можна пройти скільки разів?

<details><summary>Відповідь</summary>Лише один. Другий прохід по вичерпаному генератору не підніме помилку одразу, але поверне порожній результат -- тому для аналітики, що потребує кількох проходів, генератор матеріалізують у <code>list</code> один раз.</details>

**4.** Навіщо в пайплайні два окремі NamedTuple (`RawOrder` і `Order`), а не один?

<details><summary>Відповідь</summary><code>RawOrder</code> відповідає сирим даним із джерела (є <code>timestamp</code>); <code>Order</code> -- уже результат трансформації (є <code>day</code>/<code>time</code>, <code>timestamp</code> більше не потрібен). Розділення робить кожен крок пайплайна явним і тестованим окремо.</details>

**5.** Якою мовою (в термінах цього уроку) описати різницю generator-пайплайну й eager-пайплайну на списках?

<details><summary>Відповідь</summary>Обидва дають ІДЕНТИЧНИЙ результат (перевірено <code>assert</code>-ом) -- різниця лише в тому, коли і скільки пам'яті витрачається під час обчислення: generator -- ліниво, по одному елементу; eager (list) -- одразу все.</details>

## Далі

Урок 11 (практикум П2, «Пошук») продовжує тему ефективності — тепер на прикладі того, як властивості даних (відсортованість, суміжність) визначають вибір алгоритму пошуку. Урок 12 («Модулі та стандартна бібліотека») — попередній логічний крок цієї ж лінії: там `datetime`/`collections` уже застосовувались до менших обсягів того самого ресторанного датасету, перш ніж тут вони масштабувались до 500 000 рядків через генератори.